In [17]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [18]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [20]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [21]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation datGenera un conjunto de datos de evaluación para evaluar *prompts*. Este conjunto de datos se utilizará para evaluar *prompts* que generen código Python, JSON o expresiones regulares (Regex) específicamente para tareas relacionadas con AWS. Genera una matriz de objetos JSON, donde cada uno represente una tarea que requiera Python, JSON o una expresión regular para completarse.

Ejemplo de salida:
```json
[
  {
    "task": "Descripción de la tarea",
  },
  ...otros
]
```

* Céntrate en tareas que puedan resolverse escribiendo una única función de Python, un único objeto JSON o una única expresión regular.
* Céntrate en tareas que no requieran escribir mucho código.

Por favor generar 3 objetos.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [22]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [23]:
def run_prompt(test_case):
    """Combina el prompt y la entrada del caso de prueba, y luego devuelve el resultado."""
    prompt = f"""
Por favor resolver la siguiente tarea:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [24]:
def run_test_case(test_case):
    """Ejecutar run_prompt luego evalúa el resultado"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [25]:
def run_eval(dataset):
    """Carga el conjunto de datos y llama a run_test_case con cada caso."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [26]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [27]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Funci\u00f3n para extraer el tipo de servicio de un ARN de AWS\n\n```python\ndef extraer_servicio_arn(arn):\n    \"\"\"\n    Extrae el tipo de servicio de un ARN de AWS.\n    \n    Formato t\u00edpico de ARN:\n    arn:partition:service:region:account-id:resource\n    arn:partition:service:region:account-id:resource-type/resource-id\n    arn:partition:service:region:account-id:resource-type:resource-id\n    \n    Args:\n        arn (str): El ARN de AWS a analizar\n    \n    Returns:\n        str: El nombre del servicio (ej: 's3', 'ec2', 'lambda') o None si es inv\u00e1lido\n    \n    Ejemplos:\n        >>> extraer_servicio_arn('arn:aws:s3:::mi-bucket')\n        's3'\n        >>> extraer_servicio_arn('arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0')\n        'ec2'\n        >>> extraer_servicio_arn('arn:aws:lambda:us-west-2:123456789012:function:mi-funcion')\n        'lambda'\n    \"\"\"\n    # Validar que el ARN no sea None o vac\u00edo\n    if not